# Módulo 7 · Clase 3 — Regresión

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | Regresión lineal simple y múltiple |
| 2 | Métricas de regresión: MAE, MSE, RMSE, R² |
| 3 | Regresión polinómica |
| 4 | Regularización: Ridge y Lasso |
| 5 | Diagnóstico con residuos |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

np.random.seed(42)
print("Librerías listas")

---
## 1. Regresión lineal

```
SIMPLE (1 feature):
  ŷ = β₀ + β₁·x₁

MÚLTIPLE (n features):
  ŷ = β₀ + β₁·x₁ + β₂·x₂ + ... + βₙ·xₙ

El modelo aprende los β minimizando:
  MSE = (1/n) · Σ(yᵢ - ŷᵢ)²

Supuestos de la regresión lineal:
  1. Linealidad entre X e y
  2. Residuos con distribución normal
  3. Homoscedasticidad (varianza de errores constante)
  4. Independencia de los residuos
  5. No multicolinealidad entre features
```

In [ ]:
# Dataset: predecir consumo de combustible
np.random.seed(42)
n = 300
peso      = np.random.normal(1500, 400, n).clip(600, 3000)
potencia  = np.random.normal(130, 50, n).clip(60, 300)
velocidad = np.random.normal(120, 30, n).clip(60, 200)
consumo   = (0.005*peso + 0.02*potencia + 0.015*velocidad
             + np.random.normal(0, 1.5, n) + 2)

df = pd.DataFrame({'peso':peso, 'potencia':potencia, 'velocidad':velocidad, 'consumo':consumo})

X = df[['peso', 'potencia', 'velocidad']]
y = df['consumo']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

modelo = LinearRegression()
modelo.fit(X_tr, y_tr)

print("=== Regresión Lineal Múltiple ===")
print(f"Intercepto (β₀): {modelo.intercept_:.4f}")
for nombre, coef in zip(X.columns, modelo.coef_):
    print(f"  β({nombre}): {coef:.6f}")

print(f"\nFórmula aprendida:")
print(f"  consumo = {modelo.intercept_:.4f}")
for nombre, coef in zip(X.columns, modelo.coef_):
    print(f"           + {coef:.5f} × {nombre}")

y_pred = modelo.predict(X_te)
print(f"\nR² en test: {r2_score(y_te, y_pred):.4f}")

---
## 2. Métricas de regresión

```
MAE  (Mean Absolute Error):  (1/n) Σ |yᵢ - ŷᵢ|
  → Error medio en las mismas unidades que y
  → Robusto a outliers
  → Fácil de interpretar: "el modelo se equivoca en ±X unidades de media"

MSE  (Mean Squared Error):   (1/n) Σ (yᵢ - ŷᵢ)²
  → Penaliza errores grandes más que los pequeños
  → Unidades al cuadrado (menos interpretable)

RMSE (Root MSE):             √MSE
  → Mismas unidades que y
  → Más sensible a outliers que MAE

R²   (Coeficiente de determinación):  1 - SS_res/SS_tot
  → Proporción de varianza explicada por el modelo [0, 1]
  → R²=1: predicción perfecta, R²=0: igual que predecir la media
  → Puede ser negativo (modelo peor que la media)
  → Depende del dataset: no comparar R² entre datasets distintos
```

In [ ]:
y_pred = modelo.predict(X_te)

mae  = mean_absolute_error(y_te, y_pred)
mse  = mean_squared_error(y_te, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_te, y_pred)

print("=== Métricas en test ===")
print(f"MAE:  {mae:.4f}  → el modelo se equivoca ±{mae:.2f} L/100km de media")
print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}  → error típico de {rmse:.2f} L/100km")
print(f"R²:   {r2:.4f}   → el modelo explica el {r2*100:.1f}% de la varianza")

# Comparar con baseline (predice siempre la media)
y_baseline = np.full(len(y_te), y_tr.mean())
print(f"\nBaseline (predice media={y_tr.mean():.2f}):")
print(f"  MAE:  {mean_absolute_error(y_te, y_baseline):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_te, y_baseline)):.4f}")
print(f"  R²:   {r2_score(y_te, y_baseline):.4f}")

# Función de métricas reutilizable
def evaluar_regresion(y_real, y_pred, nombre="Modelo"):
    mae  = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2   = r2_score(y_real, y_pred)
    print(f"{nombre:20} MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}")
    return {'mae': mae, 'rmse': rmse, 'r2': r2}

print("\n=== Tabla comparativa ===")
evaluar_regresion(y_te, y_baseline, "Baseline")
evaluar_regresion(y_te, y_pred, "LinearRegression")

---
## 3. Regresión polinómica

```
¿Qué hacer cuando la relación no es lineal?

Añadimos features polinómicas: x → [x, x², x³, ...]

PolynomialFeatures(degree=2) con features [a, b] genera:
  [1, a, b, a², a·b, b²]

El modelo sigue siendo LINEAR en los parámetros:
  ŷ = β₀ + β₁a + β₂b + β₃a² + β₄ab + β₅b²

⚠ A mayor degree → más riesgo de overfitting
```

In [ ]:
# Relación no lineal: temperatura vs ventas de helados
np.random.seed(42)
temp = np.sort(np.random.uniform(5, 40, 150))
# Relación cuadrática: ventas óptimas alrededor de 28°
ventas = -0.5*(temp - 28)**2 + 100 + np.random.normal(0, 8, 150)
ventas = ventas.clip(0)

X_t = temp.reshape(-1, 1)
X_tr, X_te, y_tr, y_te = train_test_split(X_t, ventas, test_size=0.2, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x_rango = np.linspace(5, 40, 200).reshape(-1, 1)

colores = ['steelblue', 'green', 'orange']
for ax, degree, color in zip(axes, [1, 2, 5], colores):
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('modelo', LinearRegression())
    ])
    pipe.fit(X_tr, y_tr)
    r2_tr = r2_score(y_tr, pipe.predict(X_tr))
    r2_te = r2_score(y_te, pipe.predict(X_te))

    ax.scatter(X_tr, y_tr, alpha=0.4, color='gray', s=15, label='Train')
    ax.scatter(X_te, y_te, alpha=0.7, color='red', s=20, marker='x', label='Test')
    ax.plot(x_rango, pipe.predict(x_rango), color=color, lw=2)
    ax.set_title(f'Grado {degree}\nR²_train={r2_tr:.3f}, R²_test={r2_te:.3f}')
    ax.set_xlabel('Temperatura (°C)'); ax.set_ylabel('Ventas')
    ax.legend(fontsize=8)

plt.suptitle('Regresión polinómica: grado vs overfitting', fontsize=11)
plt.tight_layout(); plt.show()
print("Grado 1: subajuste (underfitting) — curva demasiado simple")
print("Grado 2: buen ajuste — captura la forma cuadrática real")
print("Grado 5: sobreajuste (overfitting) — se ajusta al ruido del train")

---
## 4. Regularización: Ridge y Lasso

```
PROBLEMA: coeficientes muy grandes → overfitting
SOLUCIÓN: añadir una penalización a los coeficientes grandes

RIDGE (L2):  MSE + α · Σβᵢ²
  → Reduce todos los coeficientes, ninguno a exactamente 0
  → Bueno cuando todas las features son relevantes

LASSO (L1):  MSE + α · Σ|βᵢ|
  → Puede llevar coeficientes exactamente a 0 → selección de features
  → Bueno cuando solo algunas features son relevantes

α (alpha): controla la fuerza de regularización
  α=0 → igual que LinearRegression
  α→∞ → todos los coeficientes → 0
```

In [ ]:
# Dataset con muchas features, algunas irrelevantes
np.random.seed(42)
n, n_features = 200, 20
X_reg = np.random.randn(n, n_features)
# Solo las primeras 5 features son relevantes
coef_real = np.array([3, -2, 1.5, -1, 0.5] + [0]*15)
y_reg = X_reg @ coef_real + np.random.randn(n) * 0.5

X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

modelos = {
    'LinearRegression': LinearRegression(),
    'Ridge α=0.1':      Ridge(alpha=0.1),
    'Ridge α=10':       Ridge(alpha=10),
    'Lasso α=0.1':      Lasso(alpha=0.1),
    'Lasso α=1.0':      Lasso(alpha=1.0),
}

print(f"{'Modelo':20} {'R²_train':>10} {'R²_test':>10} {'Coef≠0':>8} {'Max|coef|':>12}")
resultados = {}
for nombre, m in modelos.items():
    m.fit(X_tr, y_tr)
    r2_tr = r2_score(y_tr, m.predict(X_tr))
    r2_te = r2_score(y_te, m.predict(X_te))
    n_nonzero = np.sum(np.abs(m.coef_) > 1e-6)
    max_coef  = np.max(np.abs(m.coef_))
    print(f"{nombre:20} {r2_tr:>10.4f} {r2_te:>10.4f} {n_nonzero:>8} {max_coef:>12.4f}")
    resultados[nombre] = m.coef_

# Visualizar coeficientes
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

x_idx = np.arange(n_features)
axes[0].bar(x_idx - 0.3, coef_real, 0.3, label='Real', color='black', alpha=0.7)
axes[0].bar(x_idx,       resultados['LinearRegression'], 0.3, label='LinearReg', color='steelblue', alpha=0.7)
axes[0].bar(x_idx + 0.3, resultados['Ridge α=10'], 0.3, label='Ridge α=10', color='coral', alpha=0.7)
axes[0].set_title('Coeficientes: Real vs LinearReg vs Ridge')
axes[0].legend(); axes[0].set_xlabel('Feature')

axes[1].bar(x_idx - 0.15, coef_real, 0.3, label='Real', color='black', alpha=0.7)
axes[1].bar(x_idx + 0.15, resultados['Lasso α=0.1'], 0.3, label='Lasso α=0.1', color='green', alpha=0.7)
axes[1].set_title('Lasso: selección de features (coef=0 → feature eliminada)')
axes[1].legend(); axes[1].set_xlabel('Feature')

plt.tight_layout(); plt.show()
print(f"\nLasso elimina {np.sum(np.abs(resultados['Lasso α=0.1']) < 1e-6)} features de {n_features}")

---
## 5. Diagnóstico con residuos

In [ ]:
# Residuos: diferencia entre real y predicho
y_pred_tr = modelo.predict(X_tr)
y_pred_te = modelo.predict(X_te)
residuos_tr = y_tr - y_pred_tr
residuos_te = y_te - y_pred_te

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 1. Real vs Predicho
axes[0,0].scatter(y_pred_te, y_te, alpha=0.5, color='steelblue', s=20)
lim = [min(y_te.min(), y_pred_te.min()), max(y_te.max(), y_pred_te.max())]
axes[0,0].plot(lim, lim, 'r--', lw=2, label='Perfecta')
axes[0,0].set_xlabel('Predicho'); axes[0,0].set_ylabel('Real')
axes[0,0].set_title('Real vs Predicho'); axes[0,0].legend()

# 2. Residuos vs Predicho
axes[0,1].scatter(y_pred_te, residuos_te, alpha=0.5, color='coral', s=20)
axes[0,1].axhline(0, color='red', ls='--', lw=2)
axes[0,1].set_xlabel('Predicho'); axes[0,1].set_ylabel('Residuo')
axes[0,1].set_title('Residuos vs Predicho\n(deben estar alrededor de 0)')

# 3. Histograma de residuos
from scipy import stats
axes[1,0].hist(residuos_te, bins=25, density=True, color='steelblue', alpha=0.7)
x_norm = np.linspace(residuos_te.min(), residuos_te.max(), 100)
axes[1,0].plot(x_norm, stats.norm.pdf(x_norm, residuos_te.mean(), residuos_te.std()), 'r-', lw=2)
axes[1,0].set_title('Distribución de residuos\n(deben ser normales)')

# 4. QQ-plot de residuos
stats.probplot(residuos_te, plot=axes[1,1])
axes[1,1].set_title('QQ-plot de residuos')

plt.suptitle('Diagnóstico del modelo de regresión', fontsize=11)
plt.tight_layout(); plt.show()

# Test de normalidad de residuos
_, p_shapiro = stats.shapiro(residuos_te)
print(f"Test Shapiro-Wilk en residuos: p={p_shapiro:.4f}")
print(f"Residuos {'normales ✓' if p_shapiro > 0.05 else 'NO normales ✗'} (α=0.05)")
print(f"Media de residuos: {residuos_te.mean():.6f} (debe ser ≈0)")

---
## Práctica

### Ejercicio 1 — Regresión sobre California Housing

**Enunciado:** Usa el dataset de casas de California:

```python
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()
```

1. Explora el dataset: shape, feature_names, estadísticas
2. Divide train/test (80/20)
3. Escala las features con StandardScaler (recuerda: fit solo en train)
4. Entrena LinearRegression
5. Calcula MAE, RMSE y R² en test. Interpreta los resultados

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

housing = fetch_california_housing()
X, y = housing.data, housing.target  # y en $100,000

# 1. Explorar
print(f"Shape: {X.shape}")
print(f"Features: {housing.feature_names}")
print(f"Target: precio mediano de casa (×$100k). Rango: [{y.min():.2f}, {y.max():.2f}]")

# 2. Split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Escalar
sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s = sc.transform(X_te)

# 4. Entrenar
modelo = LinearRegression().fit(X_tr_s, y_tr)

# 5. Evaluar
y_pred = modelo.predict(X_te_s)
mae  = mean_absolute_error(y_te, y_pred)
rmse = np.sqrt(mean_squared_error(y_te, y_pred))
r2   = r2_score(y_te, y_pred)

print(f"\nMAE:  {mae:.4f} (×$100k) = ${mae*100000:,.0f} de error medio")
print(f"RMSE: {rmse:.4f} (×$100k) = ${rmse*100000:,.0f} de error típico")
print(f"R²:   {r2:.4f} → el modelo explica el {r2*100:.1f}% de la varianza")
print("\nCoeficientes más relevantes:")
for feat, coef in sorted(zip(housing.feature_names, modelo.coef_),
                          key=lambda x: abs(x[1]), reverse=True):
    print(f"  {feat:20}: {coef:+.4f}")
```
</details>

---
### Ejercicio 2 — Encontrar el mejor grado polinómico

**Enunciado:** Sobre el ejemplo de temperatura-ventas de helados:

```python
np.random.seed(42)
temp   = np.sort(np.random.uniform(5, 40, 150)).reshape(-1,1)
ventas = (-0.5*(temp.ravel()-28)**2 + 100 + np.random.normal(0, 8, 150)).clip(0)
```

1. Prueba grados 1 a 8
2. Para cada grado calcula R² en train y test
3. Visualiza en un gráfico: eje x = grado, eje y = R²
4. ¿A partir de qué grado empieza el overfitting?
5. ¿Cuál es el grado óptimo? ¿Por qué?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline

np.random.seed(42)
temp   = np.sort(np.random.uniform(5, 40, 150)).reshape(-1,1)
ventas = (-0.5*(temp.ravel()-28)**2 + 100 + np.random.normal(0, 8, 150)).clip(0)

X_tr, X_te, y_tr, y_te = train_test_split(temp, ventas, test_size=0.2, random_state=42)

r2_trains, r2_tests = [], []
for d in range(1, 9):
    pipe = Pipeline([
        ('poly', PolynomialFeatures(d, include_bias=False)),
        ('reg',  LinearRegression())
    ])
    pipe.fit(X_tr, y_tr)
    r2_trains.append(r2_score(y_tr, pipe.predict(X_tr)))
    r2_tests.append(r2_score(y_te, pipe.predict(X_te)))

grados = range(1, 9)
plt.figure(figsize=(10, 4))
plt.plot(grados, r2_trains, 'b-o', label='R² train')
plt.plot(grados, r2_tests,  'r-o', label='R² test')
plt.axvline(2, color='green', ls='--', alpha=0.7, label='Grado óptimo=2')
plt.xlabel('Grado polinómico'); plt.ylabel('R²')
plt.title('R² en train y test por grado polinómico')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

mejor_grado = np.argmax(r2_tests) + 1
print(f"Grado con mayor R² en test: {mejor_grado}")
print(f"R²_train={r2_trains[mejor_grado-1]:.4f}, R²_test={r2_tests[mejor_grado-1]:.4f}")
print("\nA partir del grado 3-4, R² train sigue subiendo pero test baja → overfitting")
print("El grado 2 captura la relación cuadrática real. Más grado = ajustar ruido.")
```
</details>

---
### Ejercicio 3 — Ridge vs Lasso: ¿cuándo usar cada uno?

**Enunciado:** Genera dos datasets:

```python
np.random.seed(42)
n, p = 100, 30
X = np.random.randn(n, p)

# Dataset A: todas las features importan (pequeños coeficientes)
coef_A = np.random.randn(p) * 0.5
y_A = X @ coef_A + np.random.randn(n)

# Dataset B: solo 5 features importan (resto son ruido)
coef_B = np.zeros(p)
coef_B[:5] = np.array([3, -2, 1.5, -1, 0.5])
y_B = X @ coef_B + np.random.randn(n)
```

Para cada dataset, compara Ridge α=1 y Lasso α=0.1 con LinearRegression. ¿Qué modelo gana en cada caso? ¿Por qué?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score

np.random.seed(42)
n, p = 100, 30
X = np.random.randn(n, p)

coef_A = np.random.randn(p) * 0.5
y_A = X @ coef_A + np.random.randn(n)

coef_B = np.zeros(p)
coef_B[:5] = [3, -2, 1.5, -1, 0.5]
y_B = X @ coef_B + np.random.randn(n)

modelos = {
    'LinearRegression': LinearRegression(),
    'Ridge α=1':        Ridge(alpha=1),
    'Lasso α=0.1':      Lasso(alpha=0.1),
}

for nombre_ds, (X_ds, y_ds) in [('Dataset A (todas importan)', (X, y_A)),
                                  ('Dataset B (solo 5)',        (X, y_B))]:
    print(f"\n=== {nombre_ds} ===")
    X_tr, X_te, y_tr, y_te = train_test_split(X_ds, y_ds, test_size=0.3, random_state=42)
    for nombre_m, m in modelos.items():
        m.fit(X_tr, y_tr)
        r2_tr = r2_score(y_tr, m.predict(X_tr))
        r2_te = r2_score(y_te, m.predict(X_te))
        n_cero = np.sum(np.abs(m.coef_) < 1e-6)
        print(f"  {nombre_m:20} R²_tr={r2_tr:.3f} R²_te={r2_te:.3f} coef_cero={n_cero}")

print("\nConclusión:")
print("  Dataset A: Ridge gana — todos los coeficientes son pequeños pero no cero.")
print("  Dataset B: Lasso gana — pone exactamente a 0 las 25 features irrelevantes.")
```
</details>

---
### Ejercicio 4 — Diagnóstico de residuos

**Enunciado:** Genera un dataset donde LinearRegression tiene problemas (heterocedasticidad):

```python
np.random.seed(42)
x = np.sort(np.random.uniform(1, 100, 200))
# El error crece con x (heterocedasticidad)
y = 2*x + np.random.randn(200) * x * 0.2
```

1. Entrena LinearRegression y calcula R²
2. Haz el diagnóstico de residuos (4 gráficas)
3. ¿Qué problema detectas?
4. Aplica transformación log: `y_log = np.log(y)`, entrena de nuevo y compara los residuos

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

np.random.seed(42)
x = np.sort(np.random.uniform(1, 100, 200))
y = 2*x + np.random.randn(200) * x * 0.2

def diagnostico(X, y, titulo):
    X_tr, X_te, y_tr, y_te = train_test_split(X.reshape(-1,1), y, test_size=0.2, random_state=42)
    m = LinearRegression().fit(X_tr, y_tr)
    y_pred = m.predict(X_te)
    res = y_te - y_pred
    r2  = r2_score(y_te, y_pred)

    fig, axes = plt.subplots(1, 3, figsize=(13, 3))
    fig.suptitle(f'{titulo} (R²={r2:.4f})')

    axes[0].scatter(y_pred, res, alpha=0.5, s=15, color='steelblue')
    axes[0].axhline(0, color='red', ls='--'); axes[0].set_title('Residuos vs Predicho')

    axes[1].hist(res, bins=30, color='coral', density=True)
    xn = np.linspace(res.min(), res.max(), 100)
    axes[1].plot(xn, stats.norm.pdf(xn, res.mean(), res.std()), 'b-', lw=2)
    axes[1].set_title('Distribución de residuos')

    stats.probplot(res, plot=axes[2])
    axes[2].set_title('QQ-plot')

    plt.tight_layout(); plt.show()
    return r2

r2_orig = diagnostico(x, y, 'Con y original (heterocedasticidad)')
r2_log  = diagnostico(x, np.log(y), 'Con log(y) (residuos más homogéneos)')

print(f"R² original: {r2_orig:.4f}")
print(f"R² con log:  {r2_log:.4f}")
print("\nEl QQ-plot y el gráfico de residuos vs predicho muestran:")
print("  Original: varianza de errores CRECE con x (heterocedasticidad)")
print("  log(y): residuos más uniformes → supuesto de homoscedasticidad cumplido")
```
</details>

---
### Ejercicio 5 (reto) — Selección de α óptimo con validación

**Enunciado:** Implementa búsqueda manual del mejor alpha para Ridge y Lasso:

1. Genera un dataset con 15 features (5 relevantes, 10 ruido)
2. Prueba alphas: `[0.001, 0.01, 0.1, 1, 10, 100, 1000]`
3. Para cada alpha, calcula R² en train y test
4. Visualiza las curvas de Ridge y Lasso
5. ¿Cuál es el alpha óptimo para cada modelo? ¿Por qué divergen?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

np.random.seed(42)
n, p = 150, 15
X = np.random.randn(n, p)
coef = np.array([2, -1.5, 1, -0.8, 0.5] + [0]*10)
y = X @ coef + np.random.randn(n) * 0.5

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1000]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, ModelClass, nombre in zip(axes, [Ridge, Lasso], ['Ridge', 'Lasso']):
    r2_tr_list, r2_te_list = [], []
    for alpha in alphas:
        m = ModelClass(alpha=alpha).fit(X_tr, y_tr)
        r2_tr_list.append(r2_score(y_tr, m.predict(X_tr)))
        r2_te_list.append(r2_score(y_te, m.predict(X_te)))

    mejor_idx = np.argmax(r2_te_list)
    ax.semilogx(alphas, r2_tr_list, 'b-o', label='R² train')
    ax.semilogx(alphas, r2_te_list, 'r-o', label='R² test')
    ax.axvline(alphas[mejor_idx], color='green', ls='--',
               label=f'α óptimo={alphas[mejor_idx]}')
    ax.set_xlabel('Alpha (log)'); ax.set_ylabel('R²')
    ax.set_title(f'{nombre}: R² vs Alpha')
    ax.legend(); ax.grid(True, alpha=0.3)
    print(f"{nombre}: mejor alpha={alphas[mejor_idx]}, R²_test={r2_te_list[mejor_idx]:.4f}")

plt.tight_layout(); plt.show()
print("\nConclusión:")
print("  Lasso necesita alpha menor para no eliminar los coeficientes relevantes.")
print("  Ridge puede tolerar alphas mayores — solo encoge, no elimina.")
print("  En C7 veremos GridSearchCV para encontrar alpha de forma automática.")
```
</details>

---
## Resumen

```python
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Métricas de regresión
mae  = mean_absolute_error(y_real, y_pred)
rmse = np.sqrt(mean_squared_error(y_real, y_pred))
r2   = r2_score(y_real, y_pred)

# Regresión polinómica
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_train)

# Regularización
Ridge(alpha=1.0)   # L2: reduce coeficientes, no los anula
Lasso(alpha=0.1)   # L1: puede anular coeficientes (selección)
```

## ¿Qué viene en la próxima clase?
En la **Clase 4** pasamos a clasificación con Regresión Logística y aprendemos las métricas específicas para problemas de clasificación.